# NB03 — Phase Decoding（成個系統嘅心臟）
由捕捉返嚟嘅 9 張圖，逐步還原**絕對相位圖**：每個 pixel 一個數，
直接話你知佢俾投影儀邊一列照住。呢個係成條 pipeline 最精密嘅部分。
本課每步都同 C++ 實作 pixel-exact 對照（用 gtest 入面嘅 golden values）。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'


## Step 1: wrapped phase（相位主值）
$$\phi = -\operatorname{atan2}\Big(\sum_k I_k\sin\tfrac{2\pi k}{N},\ \sum_k I_k\cos\tfrac{2\pi k}{N}\Big)$$
N=4 步相位平移 → 每個 pixel 一個 [−π, π] 嘅相位（鋸齒波，每週期 reset）。

In [2]:
from sl_edu import decode, oracle

imgs = oracle.load_shift_graycode(ROOT)
wrapped = decode.wrapped_phase(imgs[:4], shift_time=4)
conf = decode.confidence_map(imgs[:4])

plt.imshow(wrapped, cmap='twilight', vmin=-np.pi, vmax=np.pi)
plt.colorbar(label='wrapped phase (rad)'); plt.title('wrapped phase'); plt.show()

plt.plot(wrapped[540, 400:640])
plt.ylabel('phase (rad)'); plt.xlabel('column')
plt.title('sawtooth: wraps from +pi to -pi at each period boundary'); plt.show()

/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82687/2441232218.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.colorbar(label='wrapped phase (rad)'); plt.title('wrapped phase'); plt.show()
/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82687/2441232218.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('sawtooth: wraps from +pi to -pi at each period boundary'); plt.show()


## Step 2: gray code → floor map
格雷碼俾每個 pixel 一個「週期門牌」：binary 解碼用 running XOR（MSB 先行）。
但格雷邊界（中週期）同相位 wrap 邊界（週期頭）唔對齊——需要修正：
- 每個 floor 區域搵 |φ| 最接近 π 嘅列 = 真正週期邊界 `mid`
- `(|φ| < 2π/3 且 j < mid) 或 φ ≥ 2π/3` → floor 減 1

In [3]:
floor_map = decode.floor_map(imgs[4:], conf, wrapped, n_periods=32, threshold=5.0)

plt.imshow(floor_map, cmap='viridis'); plt.colorbar(label='period index')
plt.title('floor map (which period)'); plt.show()

# 驗證同 C++ 一致（gtest golden values, threshold=70）
floor70 = decode.floor_map(imgs[4:], conf, wrapped, 32, 70.0)
assert floor70[453][700] == 17, 'C++ golden value mismatch!'
print('C++ golden anchor floor[453][700]==17  ->  Python matches')

C++ golden anchor floor[453][700]==17  ->  Python matches


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82687/1979812127.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('floor map (which period)'); plt.show()


## Step 3: unwrap
$$\Phi = \phi_{wrapped} + 2\pi\cdot floor + \pi$$
+π 係將範圍平移到由 0 開始（同 C++ 一致）。低信心 pixel → 0（無效）。

In [4]:
absolute = decode.unwrap(wrapped, floor_map, conf, threshold=5.0)

plt.imshow(absolute, cmap='inferno'); plt.colorbar(label='absolute phase (rad)')
plt.title('absolute phase = projector column address'); plt.show()

# 另一個 C++ golden anchor（threshold=70）
abs70 = decode.unwrap(wrapped, floor70, conf, 70.0)
assert abs(abs70[460][653] - 103.75) <= 0.1
print(f'C++ golden anchor unwrap[460][653]=103.75  ->  Python {abs70[460][653]:.3f}')

C++ golden anchor unwrap[460][653]=103.75  ->  Python 103.751


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82687/348342345.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('absolute phase = projector column address'); plt.show()


## 教學位：wrap 邊界嘅 ±1px tie
喺相位啱啱 = π 嘅邊界列，atan2 個分子數值上係零，正負由 uint8 量化決定——
等於擲毫。C++ 同 Python 喺呢啲列可以差 1px floor。**唔係 bug，係數值本質**。
（我哋嘅 test 特登放寬呢一格，仲註明咗原因。）

## 右邊緣 artifact
平移格雷碼會 wrap-around：最後 30 列解到做第 0 週期。C++ 一樣有。
實戰上 crop 邊緣或靠 confidence/disparity 過濾。

下一課：標定——部相機同投影儀點樣「認識」彼此。